In [2]:
# ============================================================
# CELL 1 — IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import time

from scipy.sparse import hstack

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Libraries imported successfully.")


Libraries imported successfully.


In [3]:
# ============================================================
# CELL 2 — LOAD MASTER DATASET
# ============================================================

file_path = r"Output\Marketing_Campaign_Feature_Engineered_Master.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)


Dataset loaded successfully.
Shape: (166665, 25)


In [4]:
# ============================================================
# CELL 3 — DATE FEATURE ENGINEERING
# ============================================================

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["DayOfWeek"] = df["Date"].dt.dayofweek

print("Date feature engineering completed.")

display(df[["Date", "Year", "Month", "DayOfWeek"]].head())


Date feature engineering completed.


,Date,Year,Month,DayOfWeek
0,2025-04-29,2025.0,4.0,1.0
1,2025-04-06,2025.0,4.0,6.0
2,2025-01-14,2025.0,1.0,1.0
3,2025-06-04,2025.0,6.0,2.0
4,2024-12-29,2024.0,12.0,6.0


In [5]:
# ============================================================
# CELL 4 — DEFINE TARGET
# ============================================================

y = df["Revenue"].copy()

print("Target variable: Revenue")
print("Target shape:", y.shape)
print("Missing target values:", y.isna().sum())


Target variable: Revenue
Target shape: (166665,)
Missing target values: 0


In [6]:
# ============================================================
# CELL 5 — DEFINE NUMERICAL X FEATURES
# ============================================================
# Excluded:
# - Revenue       -> target
# - Campaign_ID   -> identifier
# - original ROI  -> not used
# - Profit        -> direct Revenue leakage
# - Profit_Flag   -> derived from Profit
#
# Calculated_ROI is retained as confirmed by the mentor.

numerical_features = [
    "Duration",
    "Impressions",
    "Clicks",
    "Leads",
    "Conversions",
    "Acquisition_Cost",
    "Engagement_Score",
    "Calculated_ROI",
    "Channel_Google",
    "Channel_WhatsApp",
    "Channel_YouTube",
    "Channel_Email",
    "Channel_Instagram",
    "Channel_Facebook",
    "Year",
    "Month",
    "DayOfWeek"
]

print("Numerical X features:")
for col in numerical_features:
    print("-", col)

print("\nTotal numerical features:", len(numerical_features))


Numerical X features:
- Duration
- Impressions
- Clicks
- Leads
- Conversions
- Acquisition_Cost
- Engagement_Score
- Calculated_ROI
- Channel_Google
- Channel_WhatsApp
- Channel_YouTube
- Channel_Email
- Channel_Instagram
- Channel_Facebook
- Year
- Month
- DayOfWeek

Total numerical features: 17


In [7]:
# ============================================================
# CELL 6 — DEFINE CATEGORICAL X FEATURES
# ============================================================

categorical_features = [
    "Campaign_Type",
    "Target_Audience",
    "Channel_Used",
    "Language",
    "Customer_Segment",
    "Company_Name"
]

print("Categorical X features:")
for col in categorical_features:
    print("-", col)

print("\nTotal categorical features:", len(categorical_features))


Categorical X features:
- Campaign_Type
- Target_Audience
- Channel_Used
- Language
- Customer_Segment
- Company_Name

Total categorical features: 6


In [8]:
# ============================================================
# CELL 7 — CREATE X MATRIX
# ============================================================

all_features = numerical_features + categorical_features

X = df[all_features].copy()

print("X shape:", X.shape)
print("Total X columns:", X.shape[1])

print("\nX columns:")
print(X.columns.tolist())


X shape: (166665, 23)
Total X columns: 23

X columns:
['Duration', 'Impressions', 'Clicks', 'Leads', 'Conversions', 'Acquisition_Cost', 'Engagement_Score', 'Calculated_ROI', 'Channel_Google', 'Channel_WhatsApp', 'Channel_YouTube', 'Channel_Email', 'Channel_Instagram', 'Channel_Facebook', 'Year', 'Month', 'DayOfWeek', 'Campaign_Type', 'Target_Audience', 'Channel_Used', 'Language', 'Customer_Segment', 'Company_Name']


In [9]:
# ============================================================
# CELL 8 — TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Train/Test split completed.")
print("\nTraining rows:", X_train.shape[0])
print("Testing rows :", X_test.shape[0])


Train/Test split completed.

Training rows: 133332
Testing rows : 33333


In [10]:
# ============================================================
# CELL 9 — HANDLE MODEL-INPUT MISSING VALUES
# ============================================================
# Numerical missing values are filled using medians calculated
# from training data only, preventing data leakage.
#
# Categorical missing values are assigned "Unknown".
# This is only model-input handling; the approved dataset
# preprocessing was already completed before consolidation.

X_train = X_train.copy()
X_test = X_test.copy()

train_medians = X_train[numerical_features].median()

for col in numerical_features:
    X_train[col] = X_train[col].fillna(train_medians[col])
    X_test[col] = X_test[col].fillna(train_medians[col])

for col in categorical_features:
    X_train[col] = X_train[col].fillna("Unknown")
    X_test[col] = X_test[col].fillna("Unknown")

print("Model-input missing values handled.")
print("Training missing values:", X_train.isna().sum().sum())
print("Testing missing values :", X_test.isna().sum().sum())


Model-input missing values handled.
Training missing values: 0
Testing missing values : 0


In [11]:
# ============================================================
# CELL 10 — ONE-HOT ENCODING
# ============================================================

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

X_train_cat = encoder.fit_transform(X_train[categorical_features])
X_test_cat = encoder.transform(X_test[categorical_features])

print("Categorical encoding completed.")
print("Encoded training categorical shape:", X_train_cat.shape)
print("Encoded testing categorical shape :", X_test_cat.shape)


Categorical encoding completed.
Encoded training categorical shape: (133332, 178)
Encoded testing categorical shape : (33333, 178)


In [12]:
# ============================================================
# CELL 11 — COMBINE NUMERICAL + CATEGORICAL FEATURES
# ============================================================

X_train_num = X_train[numerical_features].values
X_test_num = X_test[numerical_features].values

X_train_final = hstack([
    X_train_num,
    X_train_cat
])

X_test_final = hstack([
    X_test_num,
    X_test_cat
])

print("Final model matrix created.")
print("\nTraining matrix shape:", X_train_final.shape)
print("Testing matrix shape :", X_test_final.shape)


Final model matrix created.

Training matrix shape: (133332, 195)
Testing matrix shape : (33333, 195)


In [13]:
# ============================================================
# CELL 12 — CREATE FINAL GRADIENT BOOSTING MODEL
# ============================================================

gb_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)

print("Final Gradient Boosting model created.")

print("\nConfiguration:")
print("n_estimators :", 100)
print("learning_rate:", 0.1)
print("max_depth    :", 5)
print("random_state :", 42)


Final Gradient Boosting model created.

Configuration:
n_estimators : 100
learning_rate: 0.1
max_depth    : 5
random_state : 42


In [14]:
# ============================================================
# CELL 13 — TRAIN FINAL MODEL
# ============================================================

start_time = time.time()

print("Starting Gradient Boosting training...")
print("Please wait...")

gb_model.fit(
    X_train_final,
    y_train
)

training_time_gb = time.time() - start_time

print("\nGradient Boosting training completed!")
print(f"Training time: {training_time_gb:.2f} seconds")


Starting Gradient Boosting training...
Please wait...

Gradient Boosting training completed!
Training time: 250.10 seconds


In [15]:
# ============================================================
# CELL 14 — PREDICT REVENUE
# ============================================================

y_pred_gb = gb_model.predict(X_test_final)

print("Revenue prediction completed.")
print("Number of predictions:", len(y_pred_gb))


Revenue prediction completed.
Number of predictions: 33333


In [16]:
# ============================================================
# CELL 15 — FINAL MODEL EVALUATION
# ============================================================

mae_gb = mean_absolute_error(y_test, y_pred_gb)
mse_gb = mean_squared_error(y_test, y_pred_gb)
rmse_gb = np.sqrt(mse_gb)
r2_gb = r2_score(y_test, y_pred_gb)

print("=" * 70)
print("FINAL MODEL — GRADIENT BOOSTING REGRESSOR")
print("=" * 70)

print(f"MAE  : {mae_gb:,.2f}")
print(f"MSE  : {mse_gb:,.2f}")
print(f"RMSE : {rmse_gb:,.2f}")
print(f"R²   : {r2_gb:.4f}")

print("=" * 70)


FINAL MODEL — GRADIENT BOOSTING REGRESSOR
MAE  : 10,824.80
MSE  : 332,778,151.04
RMSE : 18,242.21
R²   : 0.9986


## Final Model Result

The final selected model is **Gradient Boosting Regressor**.

The benchmark obtained during model testing was approximately:

| Metric | Result |
|---|---:|
| MAE | 10,824.80 |
| MSE | 332,778,151.04 |
| RMSE | 18,242.21 |
| R² | 0.9986 |

The notebook intentionally contains **no Linear Regression, Random Forest, or diagnostic-model sections**. Gradient Boosting is the frozen final model for this project.


In [17]:
# ============================================================
# CELL 16 — SAVE FINAL REGRESSION MODEL + PREPROCESSING
# ============================================================

import pickle
import os

# Output folder
output_folder = r"D:\Data Science\vscode\Marketing_Campaign_Performance_Prediction\Output"

# ------------------------------------------------------------
# 1. Save Gradient Boosting Regression Model
# ------------------------------------------------------------

regression_model_path = os.path.join(
    output_folder,
    "regression_gradient_boosting_model.pkl"
)

with open(regression_model_path, "wb") as file:
    pickle.dump(gb_model, file)


# ------------------------------------------------------------
# 2. Save Regression One-Hot Encoder
# ------------------------------------------------------------

regression_encoder_path = os.path.join(
    output_folder,
    "regression_encoder.pkl"
)

with open(regression_encoder_path, "wb") as file:
    pickle.dump(encoder, file)


# ------------------------------------------------------------
# 3. Save Training Medians
# ------------------------------------------------------------

regression_medians_path = os.path.join(
    output_folder,
    "regression_train_medians.pkl"
)

with open(regression_medians_path, "wb") as file:
    pickle.dump(train_medians, file)


# ------------------------------------------------------------
# 4. Save Feature Lists
# ------------------------------------------------------------

regression_features_path = os.path.join(
    output_folder,
    "regression_feature_lists.pkl"
)

feature_lists = {
    "numerical_features": numerical_features,
    "categorical_features": categorical_features,
    "all_features": all_features
}

with open(regression_features_path, "wb") as file:
    pickle.dump(feature_lists, file)


# ------------------------------------------------------------
# Confirmation
# ------------------------------------------------------------

print("=" * 70)
print("REGRESSION MODEL AND PREPROCESSING SAVED SUCCESSFULLY!")
print("=" * 70)

print("\nModel   :", regression_model_path)
print("Encoder :", regression_encoder_path)
print("Medians :", regression_medians_path)
print("Features:", regression_features_path)

print("=" * 70)

REGRESSION MODEL AND PREPROCESSING SAVED SUCCESSFULLY!

Model   : D:\Data Science\vscode\Marketing_Campaign_Performance_Prediction\Output\regression_gradient_boosting_model.pkl
Encoder : D:\Data Science\vscode\Marketing_Campaign_Performance_Prediction\Output\regression_encoder.pkl
Medians : D:\Data Science\vscode\Marketing_Campaign_Performance_Prediction\Output\regression_train_medians.pkl
Features: D:\Data Science\vscode\Marketing_Campaign_Performance_Prediction\Output\regression_feature_lists.pkl
